In [3]:
# Load ipython methods for module reloading at runtime.  Useful if a module
# gets updated between running of this notebook.
%load_ext autoreload
%autoreload 2



    
# Load my custom module paths into path.  Check to see if they are already
# in the path; if not, add them.
customModulePath = ['/home/mchurchf/pythonScripts-2024-05/WindEnergyTools',
                    '/home/mchurchf/pythonScripts-2024-05/amr-wind_tools']
import sys                
for i in range(len(customModulePath)):
    containsCustomModule = False
    for j in range(len(sys.path)):
        if (sys.path[j] == customModulePath[i]):
            containsCustomModule = True
    if not containsCustomModule:
        sys.path.append(customModulePath[i])




# Load standard modules.
import numpy as np
import pathlib
import amrex.space3d as amr
import native_boundary_plane as nbp





The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
class PoiseuilleFlow:
    """One dimensional parabolic Poiseuille functor."""

    def __init__(self, dir, D, U, profileFile):
        self.dir = dir 
        self.D = D
        self.U = U
        self.profileFile = profileFile

    
    def __call__(self, xg, yg, zg, time):        
        assert xg.shape == yg.shape
        assert xg.shape == zg.shape
        
        coord = np.zeros(np.shape(xg))
        if ((self.dir == 0) or (self.dir == 'x')):
            coord = xg
        elif ((self.dir == 1) or (self.dir == 'y')):
            coord = yg
        elif ((self.dir == 2) or (self.dir == 'z')):
            coord = zg
        
        field = np.zeros(np.shape(coord))
        
        field_ = field.flat
        coord_ = coord.flat
        for i in range(len(field_)):
            if ((coord_[i] >= 0.0) and (coord_[i] <= self.D)):
                field_[i] = 4.0 * self.U * ((coord_[i]/self.D) - (coord_[i]/self.D)**2)
            else:
                field_[i] = 0.0

        self.profile(coord, self.profileFile)

        return field

    
    def profile(self, coord, profileFile):


        coord = np.sort(np.unique(coord))
        field = np.zeros(np.shape(coord))

        fid = open(profileFile,'w')
        
        for i in range(len(coord)):
            if ((coord[i] >= 0.0) and (coord[i] <= self.D)):
                field[i] = 4.0 * self.U * ((coord[i]/self.D) - (coord[i]/self.D)**2)
            else:
                field[i] = 0.0

            fid.write(f"{coord[i]:.16e} {field[i]:.16e} 0.0 0.0 0.0\n")
        
        fid.close()


        



In [6]:
class Constant:
    """Constant functor."""

    def __init__(self, constant):
        self.constant = constant

    def __call__(self, xg, yg, zg, time):
        assert xg.shape == yg.shape
        assert xg.shape == zg.shape
        
        return self.constant * np.ones(xg.shape)




In [9]:


inputFile = '/scratch/mchurchf/tteTesting/PoiseuilleFlow/aligned-ibfm-64/poiseuille.inp'
boundaryDir = '/scratch/mchurchf/tteTesting/PoiseuilleFlow/aligned-ibfm-64/bndry_files'
profileFile = '/scratch/mchurchf/tteTesting/PoiseuilleFlow/aligned-ibfm-64/initProfile.dat'


tMin = 0.0
tMax = 100000.0
nSteps = 1
D = 1.0
U = 1.0
nu = 0.01
Re = U * D / nu


# Field to write.
field = "velocity"

# Orientation.
ori = 0  

# Number of components
ncomp = 3





# Initialize amrex.space3d.
amr.initialize([])


# Set the path to where boundary data files should be written.  Fail if it already exists unless the overwrite option is provided.
bdir = pathlib.Path(boundaryDir)
overwrite = True

if bdir.exists() and not overwrite:
    print('Boundary data already exists')




# Arrange the boundary data times to be computed/written.

steps = [i for i in range(nSteps + 1)]
times = np.linspace(tMin, tMax, nSteps + 1)

functors = {
    "velocityx": PoiseuilleFlow('z',1.0,1.0,profileFile),
    "velocityy": Constant(0.0),
    "velocityz": Constant(0.0),
}

# Write the data.
for step, time in zip(steps, times):
    odir = bdir / f"bndry_output{step:05d}"
    print(f"Generating {odir}")
    bp = nbp.NativeBoundaryPlane(field, ncomp, ori, odir)
    bp.define_from_file(inputFile, step, time)
    bp.evaluate(functors)
    bp.write()
np.savetxt(bdir / "time.dat", np.c_[steps, times], fmt="%.17g")


# Finalize amrex.space3d.  If you don't do this and you run this cell again, the kernal crashes.
amr.finalize()




AMReX (25.03) finalized
Initializing AMReX (25.03)...
AMReX (25.03) initialized
Generating /scratch/mchurchf/tteTesting/PoiseuilleFlow/aligned-ibfm-64/bndry_files/bndry_output00000
Generating /scratch/mchurchf/tteTesting/PoiseuilleFlow/aligned-ibfm-64/bndry_files/bndry_output00001
Unused ParmParse Variables:
  [TOP]::ABL.bndry_file(nvals = 1)  :: [./bndry_files]
  [TOP]::ABL.bndry_file(nvals = 1)  :: [./bndry_files]
  [TOP]::ABL.bndry_io_mode(nvals = 1)  :: [1]
  [TOP]::ABL.bndry_io_mode(nvals = 1)  :: [1]
  [TOP]::ABL.bndry_output_format(nvals = 1)  :: [native]
  [TOP]::ABL.bndry_output_format(nvals = 1)  :: [native]
  [TOP]::ABL.bndry_var_names(nvals = 1)  :: [velocity]
  [TOP]::ABL.bndry_var_names(nvals = 1)  :: [velocity]
  [TOP]::ABL.initial_wind_profile(nvals = 1)  :: [true]
  [TOP]::ABL.initial_wind_profile(nvals = 1)  :: [true]
  [TOP]::ABL.perturb_temperature(nvals = 1)  :: [false]
  [TOP]::ABL.perturb_temperature(nvals = 1)  :: [false]
  [TOP]::ABL.perturb_velocity(nvals = 1

In [32]:
amr.finalize()

Unused ParmParse Variables:
  [TOP]::ABL.bndry_file(nvals = 1)  :: [./bndry_files]
  [TOP]::ABL.bndry_io_mode(nvals = 1)  :: [1]
  [TOP]::ABL.bndry_output_format(nvals = 1)  :: [native]
  [TOP]::ABL.bndry_var_names(nvals = 1)  :: [velocity]
  [TOP]::ABL.initial_wind_profile(nvals = 1)  :: [false]
  [TOP]::ABL.perturb_temperature(nvals = 1)  :: [false]
  [TOP]::ABL.perturb_velocity(nvals = 1)  :: [false]
  [TOP]::ABL.rans_1dprofile_file(nvals = 1)  :: [initialProfile.txt]
  [TOP]::ABL.temperature_heights(nvals = 2)  :: [0.0, 0.5]
  [TOP]::ABL.temperature_values(nvals = 2)  :: [300.0, 300.0]
  [TOP]::amr.blocking_factor(nvals = 1)  :: [8]
  [TOP]::geometry.is_periodic(nvals = 3)  :: [0, 1, 0]
  [TOP]::incflo.density(nvals = 1)  :: [1.0]
  [TOP]::incflo.diffusion_type(nvals = 1)  :: [2]
  [TOP]::incflo.godunov_type(nvals = 1)  :: [weno]
  [TOP]::incflo.gravity(nvals = 3)  :: [0., 0., -9.81]
  [TOP]::incflo.physics(nvals = 1)  :: [ABL]
  [TOP]::incflo.use_godunov(nvals = 1)  :: [1]
  [TOP]